In [1]:
import json
import tiktoken

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader,TextLoader,PyPDFDirectoryLoader
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_huggingface import HuggingFaceEmbeddings


/home/diabate/Bureau/supply-chain-contract-rag-agent/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_266245/389804495.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,TextLoader,PyPDFDirectoryLoader


# Importer les pdfs pour la construction de vector store

In [25]:
pdf_path = "../data/contracts"
# Load the PDF files
loader = PyPDFDirectoryLoader(pdf_path)

## Splitters text in doc or tokenization

In [26]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,
    chunk_overlap=200,
    encoding_name="cl100k_base"
)

## Tokenizer

In [27]:
chunks = loader.load_and_split(splitter)

In [28]:
len(chunks)

180

## Encoding for Hugging Face 

In [31]:
model_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2266.20it/s]


## Transorm in vector store

In [32]:
vectorstore = Chroma.from_documents(
    chunks, 
    model_embeddings,
    collection_name="contracts",
    persist_directory="../data/vectorstore_contracts_V2")

## Retriever

In [41]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 15})

## Define fonction RAG

In [42]:
from langchain_mistralai import ChatMistralAI

cle_extraite = os.getenv("CUAD_KEY")

llm = ChatMistralAI(
    model="mistral-small-latest", 
    temperature=0.0,
    api_key=cle_extraite 
)

In [43]:
prompt = """You are an expert AI assistant specializing in legal and supply chain contracts review.
Your task is to answer the user's question accurately based ONLY on the provided contract excerpts (Context).

CRITICAL INSTRUCTIONS:
1. Language: The context might be in English, but you must ALWAYS answer in French (or the language of the question). Translate the insights accurately.
2. Grounding: Rely only on the clear facts mentioned in the context. Do not invent or assume anything.
3. Citations: You MUST explicitly cite the contract name and the page number where you found the answer (e.g., "D'après le contrat [Nom], Page [X]...").
4. Strictness: If the context does not contain the specific answer to the question, state clearly that you cannot find the information in the current documents. Do not attempt to generalize.

<context>
{context}
</context>

Question: {question}

Answer (in French with exact page citations):
"""

In [44]:
def RAG(query, llm=llm, prompt_template=prompt):
    # Récupération des blocs pertinents (avec k=5 configuré plus haut)
    relevant_document_chunks = retriever.invoke(query)
    
    # Construction d'un contexte enrichi avec les métadonnées
    context_list = []
    for chunk in relevant_document_chunks:
        # On extrait le nom du fichier (en enlevant le chemin complet pour que ce soit propre)
        source_path = chunk.metadata.get("source", "Contrat Inconnu")
        source_name = source_path.split("/")[-1] 
        
        # On récupère la page (on ajoute +1 car souvent l'indexation commence à 0)
        page_num = chunk.metadata.get("page", 0) + 1
        
        # On formate le bloc avec sa source bien visible pour Mistral
        formatted_chunk = f"[Source: {source_name} - Page {page_num}]\n{chunk.page_content}"
        context_list.append(formatted_chunk)
    
    # On rassemble tous les blocs
    context_for_query = "\n\n---\n\n".join(context_list)
    
    # On remplit le prompt et on appelle Mistral
    final_prompt = prompt_template.format(context=context_for_query, question=query)
    response = llm.invoke(final_prompt)
    
    return response.content

In [45]:
# plot markdown
from IPython.display import Markdown

In [46]:
query = "What are the payment terms for the contract with supplier X?"
answer = RAG(query)
Markdown(f"**Question:** {query}\n\n**Answer:** {answer}")

**Question:** What are the payment terms for the contract with supplier X?

**Answer:** D'après les documents fournis, **aucune mention explicite d'un contrat avec un fournisseur nommé "X"** n'a été identifiée dans les extraits de contrats disponibles. Par conséquent, je ne peux pas fournir les termes de paiement pour ce fournisseur spécifique.

Si vous souhaitez que je recherche les termes de paiement dans un contrat particulier parmi ceux partagés (par exemple, Reynolds Consumer Products Inc., West Pharmaceutical Services Inc., ou Loha Company Ltd.), précisez le nom du fournisseur ou du contrat concerné.

---
*Exemple de demande clarifiée :*
*"Quels sont les termes de paiement dans le contrat ReynoldsConsumerProductsInc_20191115_S-1_EX-10.18_11896469_EX-10.18_Supply Agreement.pdf ?"*

Je pourrai alors répondre avec des citations précises (ex. : *"D'après le contrat [Nom], Page [X]..."*).

In [49]:
query = "What facilities or plants is the Carrier responsible for constructing according to the definitions?"
answer = RAG(query)
Markdown(f"**Question:** {query}\n\n**Answer:** {answer}")

**Question:** What facilities or plants is the Carrier responsible for constructing according to the definitions?

**Answer:** D'après le contrat **RangeResourcesLouisianaInc_20150417_8-K_EX-10.5_Transportation Agreement.pdf**, le Carrier est responsable de la construction des installations suivantes :

1. **Lincoln Parish Plant** : Une usine de traitement du gaz naturel à construire par le Carrier ou l'un de ses affiliés, située à Arcadia, Lincoln Parish, Louisiane.
   *(Page 4)*

2. **Mount Olive Plant** : Une autre usine de traitement du gaz naturel à construire par le Carrier ou l'un de ses affiliés, située à Ruston, Lincoln Parish, Louisiane.
   *(Page 4)*

3. **Commencement Date Facilities** :
   - Pour le contrat **RangeResourcesLouisianaInc_20150417_8-K_EX-10.5_Transportation Agreement.pdf** :
     - 27 miles de pipeline de 10 pouces entre le Lincoln Parish Plant et le point de livraison (DCP Black Lake Pipeline).
     *(Page 39, Exhibit B)*
   - Pour le contrat **PenntexMidstreamPartnersLp_20150416_S-1A_EX-10.4_Transportation Agreement.pdf** :
     - 0,9 mile de pipeline de 24 pouces entre le Lincoln Parish Plant et les points de livraison.
     - 12 miles de pipeline de 24 pouces entre le Mount Olive Plant et les points de livraison.
     *(Page 19, Exhibit A)*

4. **Post-Commencement Date Facilities** (si le Shipper en fait la demande et que le Carrier accepte) :
   - Construction de 13 miles de pipeline de 8 pouces entre le Mount Olive Plant et le Lincoln Parish Plant, ainsi qu'un point de réception au Mount Olive Plant.
   *(Page 8, Article 7.3(a) du contrat RangeResourcesLouisianaInc_20150417_8-K_EX-10.5_Transportation Agreement.pdf)*

In [52]:
query = "qu'est ce que le contrat?"
answer = RAG(query)
Markdown(f"**Question:** {query}\n\n**Answer:** {answer}")

**Question:** qu'est ce que le contrat?

**Answer:** D'après les documents fournis, le **contrat** mentionné dans les extraits est un **accord de transport** (ou *Transportation Agreement*) entre **Range Resources Louisiana Inc.** (le *Shipper*) et un *Carrier* (transporteur), régissant les conditions de transport de gaz naturel ou de produits similaires.

**Détails clés identifiés dans les documents :**
1. **Nature du contrat** :
   - Il s'agit d'un accord de transport de gaz (*Transportation Agreement*), comme indiqué dans le titre du document principal :
     *"RangeResourcesLouisianaInc_20150417_8-K_EX-10.5_9045501_EX-10.5_Transportation Agreement.pdf"* (Page 2).
   - Le contrat définit les obligations du *Carrier* (transporteur) et du *Shipper* (expéditeur), notamment en matière de livraison, de paiement et de gestion des litiges.

2. **Parties impliquées** :
   - **Range Resources Louisiana Inc.** (Shipper) : Partie qui expédie le gaz.
   - **Carrier** : Partie qui transporte le gaz (défini dans le préambule du contrat, Page 2).

3. **Objet principal** :
   - Le contrat encadre le transport de gaz (*Gas* ou *Shipper Product*) via un système de transport dédié (*Transportation System*), incluant :
     - La *Dedication* (engagement du Shipper à livrer tout son gaz résiduel au système de transport, Page 6 du contrat *PenntexMidstreamPartnersLp*).
     - Les obligations du *Carrier* (fournir des services de transport fermes ou interruptibles, Page 6 du même contrat).
     - Les conditions de force majeure (*Force Majeure*), qui suspendent les obligations en cas d'événements imprévus (Page 11 et 12 du contrat *RangeResourcesLouisianaInc*).

4. **Autres éléments clés** :
   - **Gouvernance** : Le contrat est régi par le droit de l'État du Texas (Page 19 du contrat *RangeResourcesLouisianaInc*).
   - **Résolution des litiges** : Procédure de négociation directe avant tout recours judiciaire (Article XII, Page 12-13 du contrat *RangeResourcesLouisianaInc*).
   - **Confidentialité** : Obligation de confidentialité des termes du contrat (Article XIX, Page 21 du contrat *RangeResourcesLouisianaInc*).

**Citations exactes** :
- *"This Agreement and the Exhibits and Schedules hereto constitute the entire agreement and understanding between the Parties with respect to the subject matter hereof"* (Page 19, *RangeResourcesLouisianaInc*).
- *"Carrier shall have the meaning given to such term in the preamble of this Agreement"* (Page 2, *RangeResourcesLouisianaInc*).

**Conclusion** :
Le contrat est un **accord de transport de gaz** entre Range Resources Louisiana Inc. (Shipper) et un transporteur (Carrier), régissant les modalités de livraison, de paiement, de force majeure et de résolution des litiges, sous la juridiction du Texas.

In [53]:
query = "C'est quoi  Range Resources Louisiana Inc?"
answer = RAG(query) 
Markdown(f"**Question:** {query}\n\n**Answer:** {answer}")

**Question:** C'est quoi  Range Resources Louisiana Inc?

**Answer:** D'après le contexte fourni, **Range Resources Louisiana Inc.** n'est pas directement défini dans les extraits de contrat disponibles. Cependant, les documents mentionnent des entités liées comme **PennTex North Louisiana, LLC** et **PennTex Midstream Partners, LLC**, ainsi que des références à des accords impliquant **Range Resources - Louisiana, Inc.** dans le cadre d'un **Transportation Agreement** (accord de transport).

Voici les éléments pertinents extraits des documents :

1. **Définition d'"Affiliate"** (Page 1) :
   *"Resource Development Corp. et ses filiales, le terme « Affiliate » exclut PennTex Midstream Partners, LLC et chacune de ses filiales."*
   → Cela suggère que **Range Resources Louisiana Inc.** pourrait être lié à **Resource Development Corp.** ou à ses filiales, mais sans confirmation explicite dans les extraits.

2. **Contexte des accords** (Page 1) :
   *"PennTex North Louisiana, LLC, Shipper, PennTex NLA Holdings, LLC et MRD WHR LA Midstream LLC"* sont mentionnés dans l'**AMI/MEA Agreement** (Amended and Restated Area of Mutual Interest and Midstream Exclusivity Agreement) daté du 14 avril 2015.
   → **Range Resources - Louisiana, Inc.** est cité comme source du document (8-K, 4/17/2015), mais son rôle exact n'est pas précisé dans les extraits.

**Conclusion** :
Les documents ne fournissent pas de définition explicite de **Range Resources Louisiana Inc.** en tant qu'entité juridique distincte. Ils la mentionnent uniquement comme partie liée à des accords de transport ou comme source de divulgation (8-K).

**Réponse finale** :
*"D'après les extraits disponibles, Range Resources Louisiana Inc. n'est pas défini explicitement. Les documents la mentionnent comme partie liée à des accords de transport (ex. : Transportation Agreement) ou comme source de divulgation (8-K, 4/17/2015), mais sans préciser son rôle ou sa définition juridique exacte. Voir Page 1 et Page 22 pour les références contextuelles."*